In [ ]:

from files.transformer_classes import *

warnings.filterwarnings('ignore')

# Transformer

Objective: predict change in BTC price given features

## Prepare data
Create tupled dataset


In [ ]:
data = pd.read_csv(f'../{fullDataPath('BTC')}')
daily_data = dataSetup(data, trainingColPath='../training_columns.txt')[-LIMIT:]
daily_data

In [ ]:
daily_data[RESPONSE].plot.line(title=f'{COIN} {RESPONSE.replace(RESPONSE[0], RESPONSE[0].upper())} Price', figsize=(12, 6), ylabel='Price (USD)', xlabel='Date')

In [ ]:
daily_data['gradient'].plot.hist()

In [ ]:
daily_data[RESPONSE].describe()

# Load the Transformer Model

Sequence the data to make it predict the next price

## Preprocess the data


In [ ]:
daily_data = transformerDataSetup(daily_data)
daily_data[['sequence', 'next']]

## Load the Model

### Data Setup

In [ ]:
DAYS_TO_PREDICT = TEST_DAYS
X_train, X_test, y_train, y_test, _, _ = transformerXTrainYTrain(daily_data, testSize=len(daily_data)-DAYS_TO_PREDICT)
X_train_norm, X_test_norm, y_train_norm, y_test_norm, sequence_scaler, target_scaler = normalize(X_train, X_test, y_train, y_test)
training_cols = trainingCols()

train_stuff = daily_data.loc[X_train_norm.index, training_cols]
test_stuff = daily_data.loc[X_test_norm.index, training_cols]
X_train_norm = pd.concat([X_train_norm, train_stuff], axis=1)
X_test_norm = pd.concat([X_test_norm, test_stuff], axis=1)

In [ ]:
trainingCols()

In [ ]:
# Create model with appropriate settings for Bitcoin prediction
model = BaseTransformer(
    d_model=128,
    num_heads=8,
    num_layers=4,
    output_dim=TEST_DAYS,  # Add this line - this is crucial!
    learning_rate=1e-4,
    batch_size=32,
    dropout=0.1,
    mask_value=FILL
)
model.fit(X_train_norm, y_train_norm, epochs=30, validation_data=(X_test_norm, y_test_norm))
torch.save(model, f'../models/{COIN}_model.pth')

In [ ]:
model = torch.load(f'../models/{COIN}_model.pth')

things = X_train_norm
predictions = model.predict(things)
predictions = target_scaler.inverse_transform(predictions)
predictions_df = pd.DataFrame(predictions, index=things.index, columns=[f'Day {i+1}' for i in range(predictions.shape[1])])

In [ ]:
predictions_df['close'] = y_train
predictions_df

In [ ]:
diff = predictions - y_test
plt.axhline(0.0, color='r', linestyle='--')
diff.plot.line(title='Difference on Prediction and Actual')

In [ ]:
comparison = pd.concat([y_test, predictions], axis=1)
comparison.columns = ['actual', 'prediction']
comparison.plot.line(title='Predicted vs. Actual Value on Validation Month', xlabel='Date', ylabel='Price')

In [ ]:
new = daily_data[RESPONSE].iloc[-30:]
new, next = sequence(new, len(new)-1)
new = pd.DataFrame({'sequences': [new]})
new = pd.DataFrame({""
        'sequences': normalize_sequences(new.iloc[:, 0], sequence_scaler)
    })
model.predict(new, sequence_scaler)[0, 0]

In [ ]:
price = daily_data[RESPONSE].iloc[-1]
starter = daily_data[RESPONSE].iloc[-30:]
predictions = predict_sequence(model, price, starter, sequence_scaler, sequence_length=7)
new_days = pd.date_range(start=daily_data.index[-1] + pd.Timedelta(days=1), periods=len(predictions), freq='D')
predictions_df = pd.DataFrame(predictions, index=new_days, columns=[RESPONSE])
predictions_df.plot.line()
plt.show()

In [ ]:
predictNextNDaysTransformer(daily_data, total_length=365)
